In [1]:
from sys import path

path.append("..")

import os
from pathlib import Path

os.chdir(Path.cwd().parent)

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from src.graph_utils.reading import read_metadata
from src.settings import Settings


In [3]:
metadata = read_metadata()

In [4]:

def count_elements(row_array):
    if len(row_array) == 1 and row_array[0] == -1:
        return 0
    return len(row_array)

test_data = pd.read_parquet(Settings.processed_datasets_dir / 'table.parquet')
test_data['count'] = test_data['result'].apply(count_elements)

test_data['percentage'] = test_data.apply(lambda x: x['count'] / metadata[x['dataset_name']]['graph_count'], axis=1)
test_data = test_data.reset_index(drop=True)


test_data['features'] = test_data['features'].astype('string').astype('category')
test_data.columns

Index(['features', 'dataset_name', 'result', 'count', 'percentage'], dtype='object')

In [5]:
data2d = test_data[
    (~test_data['features'].str.contains('rounded3') 
    & ~(test_data['features'].str.contains(' '))# & ~test_data['features'].str.contains('ldp'))
    & ~test_data['dataset_name'].str.contains('brec_400')
    & ~test_data['dataset_name'].str.contains('regular-2000')
    & ~(test_data['dataset_name'].str.contains(r'graph\d(?!c)') ) # for fair comparison only connected graphs

    # those are unstable
    & ~(
        test_data['features'].str.contains('algebraic_distance(?!_rounded)')
        | test_data['features'].str.contains('katz_index(?!_rounded)')
        | test_data['features'].str.contains('lss')
        | test_data['features'].str.contains('same_community')
        | test_data['features'].str.contains('spanning_edge')
        )
    )

]

data2d = data2d.pivot(index="features", columns="dataset_name", values="percentage")

In [6]:
list(data2d.index)

["['common_neigbours']",
 "['jaccard_index']",
 "['adamic_adar']",
 "['neighborhood_distance']",
 "['neighborhood_measure']",
 "['preferential_attachment']",
 "['resource_allocation']",
 "['total_nieghbors']",
 "['ari']",
 "['lds']",
 "['scan']",
 "['cn_quadrangle']",
 "['cn_triangle']",
 "['simmelian_sparsifier_np']",
 "['edge_betweenness']",
 "['edge_betweenness_normalized']",
 "['eigenvector_centrality']",
 "['closeness']",
 "['closeness_normalized']",
 "['degree_centrality']",
 "['degree_centrality_normalized']",
 "['katz_centrality']",
 "['lcc']",
 "['common_neigbours:k_graph2']",
 "['jaccard_index:k_graph2']",
 "['adamic_adar:k_graph2']",
 "['neighborhood_distance:k_graph2']",
 "['neighborhood_measure:k_graph2']",
 "['preferential_attachment:k_graph2']",
 "['resource_allocation:k_graph2']",
 "['total_nieghbors:k_graph2']",
 "['ari:k_graph2']",
 "['lds:k_graph2']",
 "['scan:k_graph2']",
 "['cn_quadrangle:k_graph2']",
 "['cn_triangle:k_graph2']",
 "['simmelian_sparsifier_np:k_graph

In [7]:
datasets = list(data2d.columns)
datasets

['chordal10',
 'chordal6',
 'chordal7',
 'chordal8',
 'chordal9',
 'eul10',
 'eul6',
 'eul7',
 'eul8',
 'eul9',
 'ge10c',
 'graph4c',
 'graph5c',
 'graph6c',
 'graph7c',
 'graph8c',
 'graph9c',
 'highlyirregular11',
 'highlyirregular12',
 'highlyirregular13',
 'highlyirregular14',
 'highlyirregular15',
 'perfect6',
 'perfect7',
 'perfect8',
 'perfect9',
 'planar_conn.5',
 'planar_conn.6',
 'planar_conn.7',
 'planar_conn.8',
 'planar_conn.9',
 'regular',
 'selfcomp12',
 'selfcomp13',
 'selfcomp9',
 'sr291467',
 'sr351668',
 'sr351899',
 'sr361446',
 'sr361566',
 'sr371889some',
 'sr401224',
 'sr65321516some']

In [8]:
def power_mean(row, p, epsilon=1e-12):
    row_safe = row + epsilon

    if p == 0:
        return np.exp(np.log(row_safe).mean())
    else:
        return np.mean(row_safe ** p) ** (1 / p)


def metrics(row: np.ndarray, difficulty_weights: np.ndarray) -> np.ndarray:
    mean_score = row.mean()

    p0_score = power_mean(row, p=0)
    p_1_score = power_mean(row, p=-1)
    p_2_score = power_mean(row, p=-2)

    # Size weighted
    sizes = np.array([
        metadata[dataset]['graph_count']
        for dataset in datasets
    ])

    weights_size = sizes / sizes.sum()
    weighted_mean_size = (row * weights_size).sum()

    # Difficulty weighted
    weighted_mean_difficulty = (
        row * difficulty_weights
    ).sum()

    results = np.array([
        mean_score,
        p0_score,
        p_1_score,
        p_2_score,
        weighted_mean_size,
        weighted_mean_difficulty
    ])

    return results

In [9]:
# A = np.random.randint(0, 101, size=(4,3))
A = 1 - data2d.to_numpy()
A


array([[2.52330220e-02, 2.58620690e-01, 1.65441176e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [8.46967747e-01, 8.62068966e-01, 8.86029412e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [8.37053470e-01, 8.62068966e-01, 7.83088235e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [1.91712541e-04, 1.72413793e-02, 7.35294118e-03, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.64325035e-04, 1.72413793e-02, 7.35294118e-03, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.64325035e-04, 1.72413793e-02, 7.35294118e-03, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]], shape=(124, 43))

In [10]:

# difficulty weighted
dataset_difficulty = 1 - A.mean(axis=0)
difficulty_weights = (
    dataset_difficulty / dataset_difficulty.sum()
)
print(difficulty_weights)

[0.02329294 0.0203869  0.02169962 0.02246007 0.02299697 0.02366054
 0.01672809 0.0199735  0.02086271 0.02229922 0.02194355 0.01924435
 0.02059019 0.02312082 0.02468678 0.02586557 0.0266141  0.01166394
 0.01747144 0.01580491 0.01714304 0.0172281  0.02407121 0.02497729
 0.02566933 0.02608556 0.02071384 0.02331672 0.02481271 0.02588579
 0.02650346 0.01775323 0.02346395 0.0231609  0.02024414 0.02966485
 0.02982054 0.02988863 0.02969185 0.02981126 0.02951048 0.02974649
 0.02947042]


In [11]:
from itertools import combinations
from tqdm import tqdm
from heapq import heapify, heappop, heappush

In [12]:
n = A.shape[0]
num_of_metrics = 6
heap_max = 6
best_outputs = [[] for _ in range(num_of_metrics)]

for indecies in tqdm(combinations(range(n), r=3)):

    to_compare = A[list(indecies)]
    outcomes = metrics(np.max(to_compare, axis=0), difficulty_weights).tolist()
    for i in range(num_of_metrics):
        heappush(best_outputs[i], (outcomes[i], indecies))
    # print(indecies, outcomes)

    if len(best_outputs[0]) > heap_max:
        for heap in best_outputs:
            heappop(heap)

best = [max(heap) for heap in best_outputs]
print(*best, sep='\n\n')



0it [00:00, ?it/s]

310124it [00:20, 15337.27it/s]

(0.9155691936842469, (15, 16, 32))

(0.8760958591072556, (9, 15, 32))

(0.8083527045954266, (9, 15, 32))

(0.685595913618946, (9, 15, 32))

(0.9598743776454698, (15, 16, 32))

(0.8949767828194962, (15, 16, 32))


In [13]:
for heap in best_outputs:
    print(*heap, sep='\n')
    print('====')

(0.9133021503332741, (15, 32, 110))
(0.9152103152887107, (14, 21, 32))
(0.9133021503332741, (15, 32, 111))
(0.9155691936842469, (15, 16, 32))
(0.9155123655386296, (14, 16, 32))
(0.9152671434343279, (15, 21, 32))
====
(0.8710560259496053, (14, 21, 32))
(0.8713270764939773, (14, 16, 32))
(0.8711127556702382, (15, 21, 32))
(0.8760146665800967, (9, 14, 32))
(0.8713838238674606, (15, 16, 32))
(0.8760958591072556, (9, 15, 32))
====
(0.7510857503024808, (9, 10, 32))
(0.7542757500535541, (9, 32, 92))
(0.7546583654922657, (9, 32, 93))
(0.7903366269737666, (4, 9, 32))
(0.8083527045954266, (9, 15, 32))
(0.8082792042019915, (9, 14, 32))
====
(0.6506980818880737, (9, 10, 32))
(0.6554208762549444, (9, 32, 93))
(0.6552099750677547, (9, 32, 92))
(0.6740962227640209, (4, 9, 32))
(0.6855515991199729, (9, 14, 32))
(0.685595913618946, (9, 15, 32))
====
(0.9594380392785335, (15, 32, 110))
(0.9597114241195461, (14, 21, 32))
(0.9594380392785335, (15, 32, 111))
(0.9597302782465125, (15, 21, 32))
(0.9598743776

In [14]:
from collections import Counter

In [15]:
trios = [trio for outputs in best_outputs for _, trio in outputs]


descriptors = [desc for outputs in best_outputs for _, trio in outputs for desc in trio]
print(Counter(descriptors))

ranking = Counter(trios)
ranking

Counter({32: 36, 15: 17, 9: 14, 14: 11, 21: 8, 16: 8, 110: 3, 111: 3, 10: 2, 92: 2, 93: 2, 4: 2})


Counter({(14, 21, 32): 4,
         (15, 16, 32): 4,
         (14, 16, 32): 4,
         (15, 21, 32): 4,
         (15, 32, 110): 3,
         (15, 32, 111): 3,
         (9, 14, 32): 3,
         (9, 15, 32): 3,
         (9, 10, 32): 2,
         (9, 32, 92): 2,
         (9, 32, 93): 2,
         (4, 9, 32): 2})

In [16]:
functions = list(data2d.index)

In [17]:
for trio in ranking:
    print(tuple(map(lambda x: functions[x].replace("['", "").replace("']", ""), trio)), ',')

('edge_betweenness_normalized', 'lds:k_graph2', 'harmonic_closeness_normalized') ,
('edge_betweenness', 'katz_centrality', 'lds:k_graph2') ,
('edge_betweenness_normalized', 'lds:k_graph2', 'harmonic_closeness') ,
('edge_betweenness_normalized', 'eigenvector_centrality', 'lds:k_graph2') ,
('edge_betweenness', 'eigenvector_centrality', 'lds:k_graph2') ,
('edge_betweenness_normalized', 'katz_centrality', 'lds:k_graph2') ,
('lds', 'edge_betweenness', 'lds:k_graph2') ,
('lds', 'edge_betweenness_normalized', 'lds:k_graph2') ,
('lds', 'scan', 'lds:k_graph2') ,
('lds', 'lds:k_graph2', 'betweenness') ,
('lds', 'lds:k_graph2', 'betweenness_normalized') ,
('neighborhood_measure', 'lds', 'lds:k_graph2') ,
